<a href="https://colab.research.google.com/github/majavier26/DSProjects/blob/main/Anomaly%20detection/Fraud_Detection_of_Bank_Transactions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
! pip install kaggle

In [24]:
from google.colab import drive
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Location


# Machine learning
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.cluster import DBSCAN

# Neural networks
import tensorflow as tf
import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten
from tensorflow.keras.metrics import Precision, Recall, CategoricalAccuracy

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


# Fraud Detection of Bank Transactions

We are using this [Kaggle dataset](https://www.kaggle.com/datasets/valakhorasani/bank-transaction-dataset-for-fraud-detection). We will be employing two different techniques for detecting fraudulent transactions: a clustering technique called `DBSCAN` and a convolutional neural network called `AutoEncoder`.

## Preprocessing data

### Obtaining data



In [4]:
# Check for my Kaggle API
if os.path.exists('~/.kaggle/') != True:
  ! mkdir ~/.kaggle/
  ! cp /content/drive/MyDrive/Kaggle_API_Credentials/kaggle.json ~/.kaggle/
  ! chmod 600 ~/.kaggle/kaggle.json
# Check if I already have the data
if os.path.exists('/content/bank-transaction-dataset-for-fraud-detection.zip') != True:
  ! kaggle datasets download valakhorasani/bank-transaction-dataset-for-fraud-detection
  ! unzip bank-transaction-dataset-for-fraud-detection.zip

Dataset URL: https://www.kaggle.com/datasets/valakhorasani/bank-transaction-dataset-for-fraud-detection
License(s): apache-2.0
Archive:  bank-transaction-dataset-for-fraud-detection.zip
  inflating: bank_transactions_data_2.csv  


### Reading data

In [5]:
data = pd.read_csv('/content/bank-transaction-dataset-for-fraud-detection.zip')
data

,TransactionID,AccountID,TransactionAmount,TransactionDate,TransactionType,Location,DeviceID,IP Address,MerchantID,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,PreviousTransactionDate
0,TX000001,AC00128,14.09,2023-04-11 16:29:14,Debit,San Diego,D000380,162.198.218.92,M015,ATM,70,Doctor,81,1,5112.21,2024-11-04 08:08:08
1,TX000002,AC00455,376.24,2023-06-27 16:44:19,Debit,Houston,D000051,13.149.61.4,M052,ATM,68,Doctor,141,1,13758.91,2024-11-04 08:09:35
2,TX000003,AC00019,126.29,2023-07-10 18:16:08,Debit,Mesa,D000235,215.97.143.157,M009,Online,19,Student,56,1,1122.35,2024-11-04 08:07:04
3,TX000004,AC00070,184.50,2023-05-05 16:32:11,Debit,Raleigh,D000187,200.13.225.150,M002,Online,26,Student,25,1,8569.06,2024-11-04 08:09:06
4,TX000005,AC00411,13.45,2023-10-16 17:51:24,Credit,Atlanta,D000308,65.164.3.100,M091,Online,26,Student,198,1,7429.40,2024-11-04 08:06:39
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2507,TX002508,AC00297,856.21,2023-04-26 17:09:36,Credit,Colorado Springs,D000625,21.157.41.17,M072,Branch,33,Doctor,109,1,12690.79,2024-11-04 08:11:29
2508,TX002509,AC00322,251.54,2023-03-22 17:36:48,Debit,Tucson,D000410,49.174.157.140,M029,Branch,48,Doctor,177,1,254.75,2024-11-04 08:11:42
2509,TX002510,AC00095,28.63,2023-08-21 17:08:50,Debit,San Diego,D000095,58.1.27.124,M087,Branch,56,Retired,146,1,3382.91,2024-11-04 08:08:39
2510,TX002511,AC00118,185.97,2023-02-24 16:24:46,Debit,Denver,D000634,21.190.11.223,M041,Online,23,Student,19,1,1776.91,2024-11-04 08:12:22


Let's check the data if there are NaN values.

In [21]:
data.describe()

,TransactionAmount,CustomerAge,TransactionDuration,LoginAttempts,AccountBalance
count,2512.000000,2512.000000,2512.000000,2512.000000,2512.000000
mean,297.593778,44.673965,119.643312,1.124602,5114.302966
std,291.946243,17.792198,69.963757,0.602662,3900.942499
min,0.260000,18.000000,10.000000,1.000000,101.250000
25%,81.885000,27.000000,63.000000,1.000000,1504.370000
50%,211.140000,45.000000,112.500000,1.000000,4735.510000
75%,414.527500,59.000000,161.000000,1.000000,7678.820000
max,1919.110000,80.000000,300.000000,5.000000,14977.990000


In [22]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2512 entries, 0 to 2511
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   TransactionID            2512 non-null   object 
 1   AccountID                2512 non-null   object 
 2   TransactionAmount        2512 non-null   float64
 3   TransactionDate          2512 non-null   object 
 4   TransactionType          2512 non-null   object 
 5   Location                 2512 non-null   object 
 6   DeviceID                 2512 non-null   object 
 7   IP Address               2512 non-null   object 
 8   MerchantID               2512 non-null   object 
 9   Channel                  2512 non-null   object 
 10  CustomerAge              2512 non-null   int64  
 11  CustomerOccupation       2512 non-null   object 
 12  TransactionDuration      2512 non-null   int64  
 13  LoginAttempts            2512 non-null   int64  
 14  AccountBalance          

There are zero NaN values in the data.

### Transforming the columns

**Columns to be removed**

- `TransactionID`, `AccountID`
- At first glance, we could also remove `DeviceID` and `MerchantID` because these are just identification of the client, however certain devices like burner phones are more likely to be used for fraudulent transactions. Criminals could also work with the bank agents, thus `MerchantID` is also important.
- I also wanted to remove `IP Address` as I thought it would be irrelevant, but it actually reveals the location of the transaction.

**Numerical variables**
- `TransactionAmount`
- `CustomerAge`
- `TransactionDuration`
- `LoginAttempts`
- `AccountBalance`
- Treatment: `StandardScaler`

**Cardinal variables**
- `TransactionType`
- `Channel`
- `CustomerOccupation`
- Treatment: `OneHotEncoder`
- `Location`
- `IP Address`
- Treatment 1: Turn the IP address into a `GeoLocation` and we can either see if the `Location` and the `GeoLocation` belong in the same country, giving us 1 in the `isSameCountry` column, and 0 if not in the same country.
- Treatment 2: Turn the IP address into a `GeoLocation` and we can get the distance between the `Location` and `GeoLocation`. If it's under 100 km, we label the column `isClose` with 1, else 0.

**Datetime variables**
- `TransactionDate`
- `PreviousTransactionDate`
- Treatment: `TimeDelta` of the two columns

We can separate the numerical columns from the categorical variables. All of our categorical variables are cardinal, and nothing is ordinal or boolean thus we can call all of our cardinal variables as categorical.

In [25]:
num_column = data[['TransactionAmount', 'CustomerAge', 'TransactionDuration', 'LoginAttempts', 'AccountBalance']]
cat_column = data[['TransactionType', 'Channel', 'CustomerOccupation']]

#### Numerical variables

As usual, we will use scale the numerical variables using `MinMaxScaler`. Even though there are no missing values, let's still impute missing values with the mean for replicability.

In [26]:
num_processor = Pipeline(
    steps=[
           ('imputer', SimpleImputer(missing_values=np.nan, strategy='mean')),
           ('scaler', MinMaxScaler())
    ]
)

In [27]:
num_processor

Pipeline(steps=[('imputer', SimpleImputer()), ('scaler', MinMaxScaler())])

##### Datetime variables

The two datetimes `TransactionDate` and `PreviousTransactionDate` don't really tell us what is going on. What matters is the interval between the two, as people don't really make bank transactions frequently. Fraudulent transactions on the other hand may have a small interval of transaction time as:
- the fraud might be accessing the victim's account after they have just accessed theirs, as they may have captured their password
- the fraud might be practicing or trying to crack the account thus multiple login attemps have been made and the interval between the two transactions is short

In [31]:
# Convert the columns to a Pandas datetime object, subtract them, and convert the timedelta into seconds
interval_transaction_sec = (pd.to_datetime(data['PreviousTransactionDate']) - pd.to_datetime(data['TransactionDate'])).dt.total_seconds()
interval_transaction_sec

,0
0,49477134.0
1,42823516.0
2,41694656.0
3,47403415.0
4,33228915.0
...,...
2507,48178913.0
2508,51201294.0
2509,38069989.0
2510,53452056.0


And let's put this into our `num_column`.

In [32]:
num_column['interval'] = interval_transaction_sec
num_column

<ipython-input-32-afe9b9d38787>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  num_column['interval'] = interval_transaction_sec


,TransactionAmount,CustomerAge,TransactionDuration,LoginAttempts,AccountBalance,interval
0,14.09,70,81,1,5112.21,49477134.0
1,376.24,68,141,1,13758.91,42823516.0
2,126.29,19,56,1,1122.35,41694656.0
3,184.50,26,25,1,8569.06,47403415.0
4,13.45,26,198,1,7429.40,33228915.0
...,...,...,...,...,...,...
2507,856.21,33,109,1,12690.79,48178913.0
2508,251.54,48,177,1,254.75,51201294.0
2509,28.63,56,146,1,3382.91,38069989.0
2510,185.97,23,19,1,1776.91,53452056.0


#### Categorical variables

The categorical variables with their dictionary are as follows:

- `TransactionType` (Debit, Card)
- `Channel` (ATM, Online, Branch)
- `CustomerOccupation` (Student, Engineer, Doctor, Retired)

Since we already know what

##### OneHotEncoding

In [28]:
card_processor = Pipeline(
    steps=[
        ('imputer', SimpleImputer(fill_value='missing', strategy='constant')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]
)

In [29]:
card_processor

Pipeline(steps=[('imputer',
                 SimpleImputer(fill_value='missing', strategy='constant')),
                ('encoder', OneHotEncoder(handle_unknown='ignore'))])

##### Location variables

We will go with the easier route and implement the `isSameCountry` column.